# 02 Train a Two-Layer GCN on the Cora Citation Network

**Project:** GNN Structural Stability Analysis  
**Supervisor:** [Professor Yixuan He](https://search.asu.edu/profile/5144530)  
**Track:** ASU STP 499 Scientific Machine Learning Research Preparation  
**Environment:** Local iMac and ASU Sol Supercomputer  
**Notebook role:** Clean baseline training before perturbation testing

## Purpose

This notebook trains a clean two-layer Graph Convolutional Network, or GCN, on the Cora citation network using PyTorch Geometric.

The goal is to establish a baseline model before testing how changes in graph structure affect model performance.

Notebook 03 will build on this baseline by testing edge perturbations such as 5%, 10%, and 20% edge deletion.

## 1. Research Framing

A Graph Neural Network can be viewed as a mapping

$$
F(G, X) \mapsto Y,
$$

where:

- $G = (V, E)$ is the graph structure,
- $X$ is the node feature matrix,
- $Y$ is the predicted node label output.

In this notebook, the graph structure is kept clean and unchanged. This gives us the reference model needed before we study structural stability.


In [1]:
import sys
import platform
import random

import numpy as np
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Python: 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]
Platform: macOS-26.5-x86_64-i386-64bit
PyTorch: 2.2.2
CUDA available: False
Device: cpu


In [3]:
import torch_geometric

print("PyTorch Geometric:", torch_geometric.__version__)

PyTorch Geometric: 2.7.0


## 2. Reproducibility Setup

We set a random seed so that training behavior is easier to compare across runs.

This does not remove all possible variation, but it improves reproducibility by making the random initialization and other random operations more consistent.


In [6]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print("Random seed set to 42")

Random seed set to 42


Because this notebook is first tested on a local Intel iMac, the device is CPU. The same seed function also supports CUDA when the notebook is later run on ASU Sol with GPU access.


## 3. Load the Cora Dataset

Cora is a citation network dataset commonly used for node classification experiments in graph machine learning.

Each node represents a scientific paper.  
Each edge represents a citation relationship between papers.  
Each node has a feature vector that describes the paper content.  
Each node also has a class label that represents the paper category.

The task is **semi-supervised node classification**. This means the model trains on a small set of labeled nodes, then predicts labels for the remaining nodes.

In [10]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="./data/Cora", name="Cora")
data = dataset[0]

print("Dataset:", dataset.name)
print("Number of graphs:", len(dataset))
print("Number of node features:", dataset.num_features)
print("Number of classes:", dataset.num_classes)

print("\nGraph summary")
print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)
print("Feature matrix shape:", data.x.shape)
print("Label vector shape:", data.y.shape)

print("\nData split")
print("Training nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

Processing...


Dataset: Cora
Number of graphs: 1
Number of node features: 1433
Number of classes: 7

Graph summary
Nodes: 2708
Edges: 10556
Feature matrix shape: torch.Size([2708, 1433])
Label vector shape: torch.Size([2708])

Data split
Training nodes: 140
Validation nodes: 500
Test nodes: 1000


Done!


## 4. Dataset Summary

The most important graph quantities for this baseline experiment are:

- number of nodes,
- number of edges,
- number of node features,
- number of prediction classes,
- training nodes,
- validation nodes,
- test nodes.

These values help confirm that the Cora dataset loaded correctly and that the model will use the standard semi-supervised split.

In [12]:
summary = {
    "nodes": data.num_nodes,
    "edges": data.num_edges,
    "node_features": dataset.num_features,
    "classes": dataset.num_classes,
    "training_nodes": int(data.train_mask.sum()),
    "validation_nodes": int(data.val_mask.sum()),
    "test_nodes": int(data.test_mask.sum()),
}

summary


{'nodes': 2708,
 'edges': 10556,
 'node_features': 1433,
 'classes': 7,
 'training_nodes': 140,
 'validation_nodes': 500,
 'test_nodes': 1000}

### Interpretation

The Cora dataset loaded successfully through PyTorch Geometric.

The graph has 2,708 nodes and 10,556 edges. Each node has a 1,433-dimensional feature vector, and each node belongs to one of 7 classes.

The training set is intentionally small, with only 140 labeled nodes. This makes Cora a useful benchmark for semi-supervised node classification, where the model must learn from a small labeled subset while using graph structure and node features to generalize to validation and test nodes.

## 5. GCN Mathematical Formulation

A Graph Convolutional Network updates each node representation by combining information from the node itself and from its neighbors.

For a basic GCN layer, the update rule can be written as:

$$
H^{(l+1)} = \sigma \left( \hat{D}^{-\frac{1}{2}} \hat{A} \hat{D}^{-\frac{1}{2}} H^{(l)} W^{(l)} \right)
$$

where:

- \(H^{(l)}\) is the node representation matrix at layer \(l\),
- \(W^{(l)}\) is the trainable weight matrix at layer \(l\),
- \(\hat{A} = A + I\) is the adjacency matrix with self-loops added,
- \(\hat{D}\) is the degree matrix of \(\hat{A}\),
- \(\sigma\) is a nonlinear activation function such as ReLU.

For this notebook, the input matrix is:

$$
X \in \mathbb{R}^{2708 \times 1433}
$$

The model produces class scores for each node:

$$
Z \in \mathbb{R}^{2708 \times 7}
$$

This means the model outputs 7 class scores for each of the 2,708 paper nodes.

In the next section, this mathematical update is implemented using PyTorch Geometric's `GCNConv` layer.


## 6. Define the Two-Layer GCN Model

The model has this structure:

$$
1433 \rightarrow 16 \rightarrow 7
$$

That means:

- 1,433 input features per node,
- 16 hidden units in the first graph convolution layer,
- 7 output classes, one for each Cora paper category.

The first GCN layer learns hidden node representations.  
The second GCN layer converts those hidden representations into class scores.

In [23]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class BaselineGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # First graph convolution layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # Dropout helps reduce overfitting during training
        x = F.dropout(x, p=0.5, training=self.training)

        # Second graph convolution layer
        x = self.conv2(x, edge_index)

        return x


model = BaselineGCN(
    in_channels=dataset.num_features,
    hidden_channels=16,
    out_channels=dataset.num_classes,
).to(device)

model

BaselineGCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 7)
)

### Interpretation

The `BaselineGCN` model was created successfully.

The first graph convolution layer maps each Cora paper from 1,433 input features into a 16-dimensional hidden representation. The second graph convolution layer maps the hidden representation into 7 output scores, one for each paper category.

This model will be trained on the clean Cora graph first. Its test accuracy will become the baseline reference for later edge perturbation experiments.

## 7. Define the Loss Function and Optimizer

The model needs two main training components:

- a loss function to measure prediction error,
- an optimizer to update the model parameters.

Because this is a multi-class node classification task, we use cross-entropy loss. For optimization, we use Adam with weight decay.

In [26]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

criterion = torch.nn.CrossEntropyLoss()

print("Optimizer:", optimizer)
print("Loss function:", criterion)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.0005
)
Loss function: CrossEntropyLoss()


### Interpretation

The optimizer and loss function were created successfully.

Adam will update the trainable parameters of the `BaselineGCN` model during training. The learning rate is set to `0.01`, and weight decay is set to `0.0005` to help reduce overfitting.

Cross-entropy loss is appropriate here because the Cora task is a multi-class node classification problem with 7 possible classes.

## 8. Define the Accuracy Function

This helper function evaluates the model on one split of the data.

The split is selected using a mask:

- `train_mask`
- `val_mask`
- `test_mask`

The function returns the proportion of correctly classified nodes in the selected split.

In [30]:
@torch.no_grad()
def accuracy(mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    predictions = logits.argmax(dim=1)

    correct = (predictions[mask] == data.y[mask]).sum().item()
    total = int(mask.sum())

    return correct / total

### Interpretation

The accuracy function switches the model into evaluation mode, makes predictions for all nodes, and measures accuracy only on the selected split.

This allows the same function to report training accuracy, validation accuracy, and test accuracy.

## 8. Training Setup

We use:

- cross-entropy loss,
- Adam optimizer,
- learning rate $0.01$,
- weight decay $5 \times 10^{-4}$,
- 200 training epochs.

The loss is computed only on the training nodes.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)

criterion = torch.nn.CrossEntropyLoss()


## 9. Train the Baseline GCN with Early Stopping

The model is trained using only the nodes selected by `train_mask`.

Validation accuracy is used to monitor generalization during training. If validation accuracy does not improve for several epochs, training stops early. This helps reduce unnecessary training and lowers the chance of overfitting.

The test set is not used to decide when to stop training. It is reserved for final evaluation after training.

In [35]:
num_epochs = 300
patience = 40

best_val_acc = 0
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

training_log = []

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    train_acc = accuracy(data.train_mask)
    val_acc = accuracy(data.val_mask)

    training_log.append({
        "epoch": epoch,
        "loss": float(loss.item()),
        "train_accuracy": train_acc,
        "validation_accuracy": val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch: {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Train: {train_acc:.4f} | "
            f"Val: {val_acc:.4f}"
        )

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch}.")
        print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}.")
        break

# Restore the best model based on validation accuracy
model.load_state_dict(best_model_state)
model = model.to(device)

Epoch: 001 | Loss: 1.9516 | Train: 0.4500 | Val: 0.3240
Epoch: 020 | Loss: 0.2401 | Train: 1.0000 | Val: 0.7600
Epoch: 040 | Loss: 0.0630 | Train: 1.0000 | Val: 0.7540

Early stopping at epoch 57.
Best validation accuracy: 0.7660 at epoch 17.


### Interpretation

The baseline GCN trained successfully on the clean Cora graph.

Training accuracy reached 100%, which is expected because the training split contains only 140 labeled nodes. The validation accuracy improved quickly and reached its best value of 0.7660 at epoch 17.

Early stopping stopped training at epoch 57 because validation accuracy did not improve for 40 consecutive epochs. The notebook then restored the model weights from epoch 17, which had the best validation performance.

This selected model will be used for final test evaluation in the next section.

## 10. Final Evaluation on the Test Set

After training, the best model selected by validation accuracy is evaluated on the test set.

The test set was not used to choose the best epoch. It is used only here to estimate final baseline performance on unseen nodes.

In [39]:
final_train_acc = accuracy(data.train_mask)
final_val_acc = accuracy(data.val_mask)
final_test_acc = accuracy(data.test_mask)

print("Final baseline results")
print(f"Best epoch: {best_epoch}")
print(f"Train accuracy: {final_train_acc:.4f}")
print(f"Validation accuracy: {final_val_acc:.4f}")
print(f"Test accuracy: {final_test_acc:.4f}")

Final baseline results
Best epoch: 17
Train accuracy: 0.9929
Validation accuracy: 0.7660
Test accuracy: 0.7830


### Interpretation

The final baseline GCN achieved a test accuracy of **0.7830** on the clean Cora graph.

The best model was selected at epoch 17 based on validation accuracy. After restoring the best validation model, the final training accuracy was 0.9929, validation accuracy was 0.7660, and test accuracy was 0.7830.

This result is reasonable for a simple two-layer GCN on Cora. It now gives us a clean reference point before testing structural perturbations in the next notebook.

In Notebook 03, this baseline test accuracy will be compared against test accuracy after controlled edge deletion, such as 5%, 10%, and 20% edge removal.

In [42]:
baseline_results = {
    "best_epoch": best_epoch,
    "train_accuracy": final_train_acc,
    "validation_accuracy": final_val_acc,
    "test_accuracy": final_test_acc,
}

baseline_results

{'best_epoch': 17,
 'train_accuracy': 0.9928571428571429,
 'validation_accuracy': 0.766,
 'test_accuracy': 0.783}

In [44]:
import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

with open(results_dir / "baseline_gcn_cora_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Saved results to results/baseline_gcn_cora_results.json")

Saved results to results/baseline_gcn_cora_results.json


## Baseline Conclusion

This notebook trained a clean two-layer GCN on the Cora citation network. The model used the standard Cora train, validation, and test masks from PyTorch Geometric.

The clean test accuracy was **0.7830**. This value will serve as the baseline reference for the next experiment, where the graph structure will be perturbed by removing a small percentage of edges.

The next notebook will study whether the GCN remains stable when the graph topology is changed.

## 11. Interpretation and Next Step

This notebook establishes the clean GCN baseline on the original Cora graph.

The result should not be interpreted as a final research conclusion. It is a reference point for the next stage of the project.

In the next notebook, we will modify the graph structure by removing a controlled percentage of edges. Then we will compare the perturbed test accuracy against this clean baseline.

A simple stability comparison will use:

$$
\Delta_{\text{accuracy}} =
\text{Accuracy}_{\text{clean}} -
\text{Accuracy}_{\text{perturbed}}
$$

A smaller value of \(\Delta_{\text{accuracy}}\) suggests that the model is more stable under the graph perturbation. A larger value suggests that the model is more sensitive to changes in graph structure.

## 12. Next Step

The next notebook is:

`03_edge_perturbation_stability_test.ipynb`

That notebook will study how the trained GCN behaves when the graph structure is changed through controlled edge deletion.

The clean test accuracy from this notebook will be used as the baseline reference. The next notebook will compare this baseline against test accuracy after removing selected percentages of edges, such as 5%, 10%, and 20%.

In [49]:
import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

baseline_results = {
    "best_epoch": int(best_epoch),
    "train_accuracy": float(final_train_acc),
    "validation_accuracy": float(final_val_acc),
    "test_accuracy": float(final_test_acc),
}

with open(results_dir / "baseline_gcn_cora_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Saved results to results/baseline_gcn_cora_results.json")

Saved results to results/baseline_gcn_cora_results.json
